# Using Natural Gradients

A GPflow 2 natural-gradient example for the final variational layer.
The natural-gradient variables remain trainable for `NaturalGradient` and are excluded from the Adam step.


In [ ]:
import numpy as np
import tensorflow as tf
import gpflow
from gpflow.likelihoods import Gaussian
from gpflow.optimizers import NaturalGradient
from gpflow.utilities import set_trainable

from doubly_stochastic_dgp.dgp import DGP

gpflow.config.set_default_float(np.float64)

X = np.linspace(-1.0, 1.0, 16)[:, None]
Y = np.sin(4.0 * X)
Z = X[::2].copy()
kernels = [gpflow.kernels.SquaredExponential(), gpflow.kernels.SquaredExponential()]
model = DGP(X, Y, Z, kernels, Gaussian(), num_samples=2, white=True)
model.likelihood.likelihood.variance.assign(0.05)

natgrad_vars = [(model.layers[-1].q_mu, model.layers[-1].q_sqrt)]
set_trainable(model.layers[-1].q_mu, False)
set_trainable(model.layers[-1].q_sqrt, False)
adam = tf.optimizers.Adam(0.01)

loss_before = model.training_loss().numpy()
NaturalGradient(gamma=0.1).minimize(model.training_loss, var_list=natgrad_vars)
with tf.GradientTape() as tape:
    loss = model.training_loss()
gradients = tape.gradient(loss, model.trainable_variables)
adam.apply_gradients(zip(gradients, model.trainable_variables))
loss_after = model.training_loss().numpy()

float(loss_before), float(loss_after)
